In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
import sqlite3
import json

In [ ]:
load_dotenv()
api_key = os.getenv("GOOGLE_API_KEY")
url = 'https://generativelanguage.googleapis.com/v1beta/openai/'
model = 'gemini-3.1-flash-lite-preview'

gemini = OpenAI(base_url=url, api_key=api_key)

In [ ]:
DB = "books.db"

with sqlite3.connect(DB) as conn:
    cursor = conn.cursor()
    cursor.execute('CREATE TABLE IF NOT EXISTS books (title TEXT PRIMARY KEY, author TEXT, price REAL)')
    conn.commit()

In [ ]:
def add_book_to_db(book):
  with sqlite3.connect(DB) as conn:
    cursor = conn.cursor()
    cursor.execute('INSERT INTO books (title, author, price) VALUES (?, ?, ?) ON CONFLICT(title) DO UPDATE SET price = ?', (book['title'].lower(), book['author'].lower(), book['price'], book['price']))

In [ ]:
books = [
  {"title": "Hunger Games", "author": "Suzanne Collins", "price": 499}, 
  {"title": "Mockingjay", "author": "Suzanne Collins", "price": 599},
  {"title": "Queen of Nothing", "author": "Holly Black", "price": 489},
  {"title": "The Palace of Illusions", "author": "Chitra Banerjee", "price": 569},
  ]

for book in books:
  add_book_to_db(book)

In [ ]:
def get_book_by_title(title):
  print(f'Tool called to get price of book with title {title}')
  with sqlite3.connect(DB) as conn:
    cursor = conn.cursor()
    cursor.execute('SELECT author, price FROM books WHERE title == ?', (title.lower(),))
    result = cursor.fetchone()
    if not result:
      return f"{title} is not available is our bookstore."
    return f"{title} by {result[0]} costs {result[1]}"

In [ ]:
print(get_book_by_title('hunger games'))
print(get_book_by_title('hunger'))

In [ ]:
def get_books_by_author(author):
  print(f'Tool called to get details of books by author {author}')
  with sqlite3.connect(DB) as conn:
    cursor = conn.cursor()
    cursor.execute('SELECT title, price FROM books WHERE author == ?', (author.lower(),))
    result = cursor.fetchall()
    if not result:
      return f"Books by author {author} are not available is our bookstore."
    return [{"title": t, "price": p} for t,p in result]

In [ ]:
get_books_by_author('suzanne collins')

In [ ]:
title_function = {
    "name": "get_book_by_title",
    "description": "Get the author and price of a book with given title",
    "parameters": {
        "type": "object",
        "properties": {
            "title": {
                "type": "string",
                "description": "The title of the book the user wants to buy from bookstore",
            },
        },
        "required": ["title"],
        "additionalProperties": False
    }
}

author_function = {
    "name": "get_books_by_author",
    "description": "Get the title and price of books by given author",
    "parameters": {
        "type": "object",
        "properties": {
            "author": {
                "type": "string",
                "description": "The author whose books the user wants to buy from bookstore",
            },
        },
        "required": ["author"],
        "additionalProperties": False
    }
}

In [ ]:
tools = [{"type": "function", "function": title_function}, {"type": "function", "function": author_function}]

In [ ]:
system_prompt = """You are a helpful assistant bot for a bookstore. Give short, polite and courteous answers. Always be accurate. When giving details, label it. If you don't know the answer, say so.
"""

In [ ]:
history = [{"role": "system", "content": system_prompt}]

In [ ]:
function_tools = {"get_book_by_title": {
        "function": get_book_by_title,
        "argument": 'title',
    },
    "get_books_by_author": {
        "function": get_books_by_author,
        "argument": 'author',
    }
}

In [ ]:
def handle_tool_call(message):
  responses = []
  for tool_call in message.tool_calls:
    print(tool_call)
    if tool_call.function.name in function_tools:
      f = function_tools[tool_call.function.name]
      arguments = json.loads(tool_call.function.arguments)
      response = f['function'](arguments.get(f['argument']))
      responses.append({
        "role": "tool",
        "content": json.dumps(response),
        "tool_call_id": tool_call.id
      })
  
  return responses

In [ ]:
def chat(message, _):
  global history
  messages = history + [{"role": "user", "content": message}]
  response = gemini.chat.completions.create(model=model, messages=messages, tools=tools)

  while response.choices[0].finish_reason == 'tool_calls':
    message = response.choices[0].message
    responses = handle_tool_call(message)
    messages.append(message)
    messages.append(responses)
    response = gemini.chat.completions.create(model=model, messages=messages, tools=tools)

  message = response.choices[0].message.content
  history = messages + [{"role": "assistant", "content": message}]
  return message
  

In [ ]:
gr.ChatInterface(fn=chat, type="messages").launch()